In [6]:
import torch
print(torch.cuda.get_device_name(0))

Tesla T4


In [6]:
# 1. Install the missing core dependency specifically
!pip install unsloth_zoo

# 2. Re-run a clean, optimized install for Llama 3.2
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.8/224.8 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 126.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninsta

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-00dger19/unsloth_e5d8fe4dab4746ea8c73e372f50c0ebd
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-00dger19/unsloth_e5d8fe4dab4746ea8c73e372f50c0ebd
  Resolved https://github.com/unslothai/unsloth.git to commit 53af4a1b3e78f2d1cac9401fe15baf8720fc2303
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
#!git clone https://github.com/karad-tanmay/sgft-demo.git
%cd sgft-demo

/content/sgft-demo


In [9]:
%rm ../sample_data/ -rf

In [2]:
from huggingface_hub import login
login()

In [4]:
with open(".env", "w") as f:
    f.write("""MODEL_NAME=meta-llama/Llama-3.2-1B
OUTPUT_DIR=outputs/llama3_1b
BATCH_SIZE=2
EPOCHS=1
LR=2e-4
MAX_LENGTH=256
""")

from dotenv import load_dotenv
import os

load_dotenv()

MODEL_NAME = os.getenv("MODEL_NAME")
OUTPUT_DIR = os.getenv("OUTPUT_DIR")


In [5]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Supports RoPE Scaling automatically
dtype = None # Auto-detect (Float16 for T4, Bfloat16 for Ampere)
load_in_4bit = True # Use 4-bit to save memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit", # Pre-quantized
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters for efficiency (only ~1-10% parameters updated)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: higher = more capacity, but more VRAM
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized at 0 for Unsloth
    bias = "none",    # Optimized at "none"
    use_gradient_checkpointing = "unsloth", # Saves massive VRAM
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.4 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [7]:
import json
from datasets import Dataset

# Load your local JSON file
# Format: [{"input": "...", "output": "..."}, ...]
with open('data/processed/sg_dataset.json', 'r') as f:
    raw_data = json.load(f)

# The prompt emphasizes problem decomposition over calculation [cite: 49, 100]
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Analyze the following math problem and provide a step-by-step solution guidance. Do not perform any numerical calculations; focus only on the logic and sequence of steps.

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    inputs  = examples["input"]
    outputs = examples["output"]
    texts = []
    for i, o in zip(inputs, outputs):
        # Format for Llama 3.2 learning [cite: 190]
        text = alpaca_prompt.format(i, o) + " <|eot_id|>"
        texts.append(text)
    return { "text" : texts, }

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100, # Adjust based on dataset size (e.g., num_train_epochs = 1)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.143162
2,2.172006
3,2.145981
4,2.198286
5,1.992225
6,1.916034
7,1.652306
8,1.646399
9,1.462176
10,1.396667


TrainOutput(global_step=100, training_loss=1.0112221658229827, metrics={'train_runtime': 98.9727, 'train_samples_per_second': 8.083, 'train_steps_per_second': 1.01, 'total_flos': 1012848304619520.0, 'train_loss': 1.0112221658229827, 'epoch': 0.26666666666666666})

In [9]:
FastLanguageModel.for_inference(model)

# Test Input
test_input = "A farm has 200 animals. 30% are cows, 50 are pigs, and the rest are goats. How many goats are there?"

inputs = tokenizer(
[
    alpaca_prompt.format(test_input, "")
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Analyze the following math problem and provide a step-by-step solution guidance. Do not perform any numerical calculations; focus only on the logic and sequence of steps.

### Input:
A farm has 200 animals. 30% are cows, 50 are pigs, and the rest are goats. How many goats are there?

### Response:
Step 1: Identify the total number of animals on the farm.
Step 2: Determine the percentage of animals that are goats by subtracting the percentages of cows and pigs from the total.
Step 3: Calculate the number of goats by multiplying the total number of animals by the percentage of goats. 


In [15]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3, # Full pass of your 3000 samples
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 3 | Total steps = 1,125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
1,0.644397
2,0.649556
3,0.622528
4,0.741701
5,0.650554
6,0.643217
7,0.597225
8,0.735941
9,0.668192
10,0.804120


TrainOutput(global_step=1125, training_loss=0.6425015402899849, metrics={'train_runtime': 905.9372, 'train_samples_per_second': 9.934, 'train_steps_per_second': 1.242, 'total_flos': 1.12510152382464e+16, 'train_loss': 0.6425015402899849, 'epoch': 3.0})

In [16]:
FastLanguageModel.for_inference(model)

# Test Input
test_input = "A farm has 200 animals. 30% are cows, 50 are pigs, and the rest are goats. How many goats are there?"

inputs = tokenizer(
[
    alpaca_prompt.format(test_input, "")
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Analyze the following math problem and provide a step-by-step solution guidance. Do not perform any numerical calculations; focus only on the logic and sequence of steps.

### Input:
A farm has 200 animals. 30% are cows, 50 are pigs, and the rest are goats. How many goats are there?

### Response:
Step 1: Identify the total number of animals on the farm.
Step 2: Determine the combined percentage of cows and pigs.
Step 3: Subtract the combined percentage from the total percentage to find the remaining percentage for goats.
Step 4: Calculate the total number of goats by applying the remaining percentage to the total population. 


In [ ]:
import os
os.kill(os.getpid(), 9)

In [10]:
from google.colab import drive
drive.mount('/content/drive')

# Define your save path
import os
save_path = "/content/drive/MyDrive/SGFT_Llama_Project"
if not os.path.exists(save_path):
    os.makedirs(save_path)

Mounted at /content/drive


In [17]:
# Save to Google Drive
model.save_pretrained(f"{save_path}/lora_model")
tokenizer.save_pretrained(f"{save_path}/lora_model")

# Save locally to Colab's temporary files (then you can manually download)
model.save_pretrained("lora_model_local")
tokenizer.save_pretrained("lora_model_local")

# Zip the local folder for easier download to your PC
!zip -r lora_model_backup.zip lora_model_local

updating: lora_model_local/ (stored 0%)
updating: lora_model_local/adapter_model.safetensors (deflated 7%)
updating: lora_model_local/chat_template.jinja (deflated 71%)
updating: lora_model_local/tokenizer.json (deflated 85%)
updating: lora_model_local/adapter_config.json (deflated 58%)
updating: lora_model_local/README.md (deflated 65%)
updating: lora_model_local/tokenizer_config.json (deflated 45%)


In [18]:
# This merges the LoRA weights into the base model and saves it
model.save_pretrained_merged(f"{save_path}/merged_model", tokenizer, save_method = "merged_16bit")

# For local backup, zip the merged folder (Warning: Large file)
!zip -r merged_model_backup.zip {save_path}/merged_model

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:37<00:00, 37.10s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:19<00:00, 79.80s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/SGFT_Llama_Project/merged_model`
updating: content/drive/MyDrive/SGFT_Llama_Project/merged_model/ (stored 0%)
updating: content/drive/MyDrive/SGFT_Llama_Project/merged_model/chat_template.jinja (deflated 71%)
updating: content/drive/MyDrive/SGFT_Llama_Project/merged_model/tokenizer_config.json (deflated 70%)
updating: content/drive/MyDrive/SGFT_Llama_Project/merged_model/tokenizer.json (deflated 85%)
updating: content/drive/MyDrive/SGFT_Llama_Project/merged_model/config.json (deflated 57%)
updating: content/drive/MyDrive/SGFT_Llama_Project/merged_model/.cache/ (stored 0%)
updating: content/drive/MyDrive/SGFT_Llama_Project/merged_model/.cache/huggingface/ (stored 0%)
updating: content/drive/MyDrive/SGFT_Llama_Project/merged_model/.cache/huggingface/CACHEDIR.TAG (deflated 24%)
updating: content/drive/MyDrive/SGFT_Llama_Project/merged_model/.cache/huggingface/.gitignore (stored 0%)
updating: content/drive/MyDrive/SGFT_Llama

In [5]:
import os
from unsloth import FastLanguageModel

# Define the path where you saved your model
# If you saved it to the root of Colab, use "lora_model_local"
# If you saved it to Drive, use "/content/drive/MyDrive/SGFT_Llama_Project/lora_model"
model_path = "lora_model_local"

if not os.path.exists(model_path):
    print(f"❌ Error: The folder '{model_path}' was not found!")
    print("Check the file explorer on the left and update 'model_path'.")
else:
    # 1. Load your Fine-Tuned 1B Guidance Model
    model_1b, tokenizer_1b = FastLanguageModel.from_pretrained(
        model_name = model_path,
        max_seq_length = 2048,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model_1b)
    print("✅ Guidance Model (1B) loaded successfully.")

    # 2. Load the 3B Base Response Model
    model_3b, tokenizer_3b = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Llama-3.2-3B-bnb-4bit",
        max_seq_length = 2048,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model_3b)
    print("✅ Response Model (3B) loaded successfully.")

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.4 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


✅ Guidance Model (1B) loaded successfully.
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-bnb-4bit as a legacy tokenizer.


✅ Response Model (3B) loaded successfully.


In [11]:
def collaborative_solve(question):
    print(f"\nProcessing Question: {question}")

    # STAGE 1: Generate Solution Guidance (1B Model)
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Analyze the following math problem and provide a step-by-step solution guidance. Do not perform any numerical calculations; focus only on the logic and sequence of steps.

### Input:
{}

### Response:
{}"""

    inputs_1b = tokenizer_1b(
        [alpaca_prompt.format(question, "")],
        return_tensors = "pt"
    ).to("cuda")

    outputs_1b = model_1b.generate(**inputs_1b, max_new_tokens = 150)
    full_output_1b = tokenizer_1b.batch_decode(outputs_1b, skip_special_tokens = True)[0]

    # Extract only the assistant's response
    guidance = full_output_1b.split("### Response:")[1].strip()
    print(f"\n--- STEP 1: Generated Guidance (1B) ---\n{guidance}")

    # STAGE 2: Solve with Response Model (3B Model)
    # Using formal chat template to avoid gibberish/symbols
    messages = [
        {"role": "system", "content": "You are a helpful assistant that solves math problems following a specific strategy."},
        {"role": "user", "content": f"Problem: {question}\n\nStrategy to follow:\n{guidance}"}
    ]

    prompt_3b = tokenizer_3b.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True
    )

    inputs_3b = tokenizer_3b([prompt_3b], return_tensors = "pt").to("cuda")

    # Use repetition penalty and sampling to ensure clean output
    outputs_3b = model_3b.generate(
        **inputs_3b,
        max_new_tokens = 512,
        repetition_penalty = 1.2,
        temperature = 0.1,
        do_sample = True
    )

    decoded_output = tokenizer_3b.batch_decode(outputs_3b, skip_special_tokens = True)[0]

    # Clean output to show only the assistant's answer
    final_answer = decoded_output.split("assistant")[-1].strip()
    return final_answer

# --- EXECUTION ---
test_q = "A farm has 200 animals. 30% are cows, 50 are pigs, and the rest are goats. How many goats are there?"
final_result = collaborative_solve(test_q)
print(f"\n--- STEP 2: Final Computed Answer (3B) ---\n{final_result}")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Processing Question: A farm has 200 animals. 30% are cows, 50 are pigs, and the rest are goats. How many goats are there?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



--- STEP 1: Generated Guidance (1B) ---
Step 1: Identify the total number of animals on the farm.
Step 2: Determine the sum of the animals identified as cows and pigs.
Step 3: Subtract the combined total of cows and pigs from the farm's total to find the number of goats.


ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

In [13]:
import os
import torch
from unsloth import FastLanguageModel

# --- CONFIGURATION ---
# 1. Your fine-tuned folder (must contain adapter_config.json)
guidance_model_path = "lora_model_local"
# 2. The base model for computation
base_model_id = "unsloth/Llama-3.2-1B-bnb-4bit"

# 1. Load Fine-Tuned 1B Guidance Model (SG-Model) [cite: 58, 88]
model_sg, tokenizer_sg = FastLanguageModel.from_pretrained(
    model_name = guidance_model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_sg)

# --- UPDATED STAGE 2: Load Base 1B Model ---
model_base, tokenizer_base = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_base)

# MANUALLY ADD THE CHAT TEMPLATE FOR THE BASE MODEL
tokenizer_base.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "{{ '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}"
    "{% elif message['role'] == 'user' %}"
    "{{ '<|start_header_id|>user<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}"
    "{% endif %}{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"
)

# --- UPDATED COLLABORATIVE INFERENCE FUNCTION ---

def collaborative_solve(question):
    print(f"\n" + "="*50)
    print(f"QUESTION: {question}")

    # STAGE 1: Generate Solution Guidance (SG) using your fine-tuned model
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Analyze the following math problem and provide a step-by-step solution guidance. Do not perform any numerical calculations; focus only on the logic and sequence of steps.

### Input:
{}

### Response:
{}"""

    inputs_sg = tokenizer_sg(
        [alpaca_prompt.format(question, "")],
        return_tensors = "pt"
    ).to("cuda")

    outputs_sg = model_sg.generate(**inputs_sg, max_new_tokens = 150)
    full_output_sg = tokenizer_sg.batch_decode(outputs_sg, skip_special_tokens = True)[0]

    guidance = full_output_sg.split("### Response:")[1].strip()
    print(f"\n--- [1B SG-MODEL] SOLUTION GUIDANCE ---\n{guidance}")

    # STAGE 2: Computation with Base Model using the manual template
    messages = [
        {"role": "system", "content": "You are a math solver. Solve the problem by following the steps provided in the strategy."},
        {"role": "user", "content": f"Problem: {question}\n\nStrategy: {guidance}"}
    ]

    prompt_base = tokenizer_base.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True
    )

    inputs_base = tokenizer_base([prompt_base], return_tensors = "pt").to("cuda")

    outputs_base = model_base.generate(
        **inputs_base,
        max_new_tokens = 512,
        repetition_penalty = 1.2,
        temperature = 0.1,
        do_sample = True
    )

    decoded_output = tokenizer_base.batch_decode(outputs_base, skip_special_tokens = True)[0]

    # Extraction fix for base models
    if "assistant" in decoded_output:
        final_answer = decoded_output.split("assistant")[-1].strip()
    else:
        final_answer = decoded_output

    return final_answer

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-bnb-4bit as a legacy tokenizer.


In [14]:
import os
import torch
from unsloth import FastLanguageModel

# --- 1. CONFIGURATION ---
guidance_model_path = "lora_model_local" # Your trained LoRA folder
base_model_id = "unsloth/Llama-3.2-1B-bnb-4bit"

# --- 2. LOAD MODELS ---
# Load Fine-Tuned 1B Guidance Model
model_sg, tokenizer_sg = FastLanguageModel.from_pretrained(
    model_name = guidance_model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_sg)

# Load Base 1B Model for Computation
model_base, tokenizer_base = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_base)

# --- 3. FIX BASE TOKENIZER (Manual Template) ---
# Base models lack a chat template; we must add it so apply_chat_template works
tokenizer_base.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "{{ '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}"
    "{% elif message['role'] == 'user' %}"
    "{{ '<|start_header_id|>user<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}"
    "{% endif %}{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"
)

# --- 4. DEFINE PIPELINE ---
def collaborative_solve(question):
    # STAGE 1: Generate Solution Guidance (SG)
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Analyze the following math problem and provide a step-by-step solution guidance. Do not perform any numerical calculations; focus only on the logic and sequence of steps.

### Input:
{}

### Response:
{}"""

    inputs_sg = tokenizer_sg([alpaca_prompt.format(question, "")], return_tensors = "pt").to("cuda")
    outputs_sg = model_sg.generate(**inputs_sg, max_new_tokens = 150)
    guidance = tokenizer_sg.batch_decode(outputs_sg, skip_special_tokens = True)[0].split("### Response:")[1].strip()

    # STAGE 2: Computation with Base Model
    messages = [
        {"role": "system", "content": "Solve this math problem step-by-step based on the provided strategy."},
        {"role": "user", "content": f"Problem: {question}\n\nStrategy: {guidance}"}
    ]

    prompt_base = tokenizer_base.apply_chat_template(messages, tokenize = False, add_generation_prompt = True)
    inputs_base = tokenizer_base([prompt_base], return_tensors = "pt").to("cuda")

    outputs_base = model_base.generate(
        **inputs_base,
        max_new_tokens = 512,
        repetition_penalty = 1.2,
        temperature = 0.1,
        do_sample = True
    )

    decoded = tokenizer_base.batch_decode(outputs_base, skip_special_tokens = True)[0]
    return guidance, decoded.split("assistant")[-1].strip()

# --- 5. TEST SUITE ---
test_examples = [
    "A farm has 200 animals. 30% are cows, 50 are pigs, and the rest are goats. How many goats are there?",
    "The cheese pizza is cut into 12 slices and the pepperoni pizza is cut into 8 slices. If Kate's 6 friends each eat 6 cheese pizza slices and 4 pepperoni pizza slices, how many pizza pies does she need to buy?",
    "Bailey starts with $100. She receives $5 weekly allowance for 8 weeks. After 8 weeks, how much money does she have in total?",
    "A classroom whiteboard is shared by 4 teachers. Each teacher has 2 lessons a day. If the board is cleaned 3 times per lesson, how many times is it cleaned in a day?"
]

for i, example in enumerate(test_examples):
    print(f"\n" + "="*40)
    print(f"TEST CASE {i+1}: {example}")
    sg, answer = collaborative_solve(example)
    print(f"\n[1B SG-MODEL GUIDANCE]:\n{sg}")
    print(f"\n[1B BASE-MODEL ANSWER]:\n{answer}")

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

...


[1B SG-MODEL GUIDANCE]:
Step 1: Determine the total number of pizza slices consumed by all friends combined.
Step 2: Calculate the number of pizza slices remaining after the friends have finished eating.
Step 3: Divide the remaining number of slices by the total number of slices in a single pizza.
Step 4: Round up to the nearest whole number to ensure everyone has at least one slice.

[1B BASE-MODEL ANSWER]:
Solution:

The answer for this question can be found here.

This solution was written as part of our free online Math Problem Solver service which helps students solve problems from their homework assignments or other sources.

TEST CASE 3: Bailey starts with $100. She receives $5 weekly allowance for 8 weeks. After 8 weeks, how much money does she have in total?


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=


[1B SG-MODEL GUIDANCE]:
Step 1: Identify the initial amount of money Bailey possesses.
Step 2: Determine the total number of weeks the allowance is received.
Step 3: Calculate the total amount of money collected from the weekly allowance.
Step 4: Combine the initial amount with the total allowance money to find the final total.

[1B BASE-MODEL ANSWER]:
Solution:

The solution above shows that there are a few steps involved when solving problems like these.

Firstly you need to identify what information needs to be included into your equation and then determine which formula or method will work best depending upon the type of question being asked.

In order to solve this particular issue we must first decide whether it's going to require an algebraic approach or not by looking at its structure. If so, use our calculator below!

## Solve This Math Problem - Example #1
### How To Find The Solution For A Given Equation?
To get started, enter some values:
\begin{align*} \textbf{(A)} & = \\

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[1B SG-MODEL GUIDANCE]:
Step 1: Determine the total number of lessons taught by all teachers in a single day.
Step 2: Calculate the total number of times the whiteboard is cleaned based on the frequency of cleaning per lesson and the total lessons per day.

[1B BASE-MODEL ANSWER]:
Solution:

The first thing to do when solving problems like these is determine what you are being asked for.

In this case we need to know how much time each person spends teaching their class. We can use that information along with other data from our school's website or ask around at your local schools to find out who teaches which classes.

We will assume there are four people (teachers) working together as follows:
Each teacher works two hours every weekday morning
They work one hour during lunch break
On Friday they clean up after themselves before going home
So if someone cleans twice daily then another cleaner needs to be hired because otherwise everyone would have to wait until Monday to get back int

In [15]:
import os
import torch
from unsloth import FastLanguageModel

# --- 1. CONFIGURATION ---
guidance_model_path = "lora_model_local" # Your fine-tuned 1B Llama model
qwen_math_model_id = "unsloth/Qwen2.5-Math-1.5B-bnb-4bit"

# --- 2. LOAD MODELS ---
# Load Fine-Tuned 1B Guidance Model (Llama-based)
model_sg, tokenizer_sg = FastLanguageModel.from_pretrained(
    model_name = guidance_model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_sg)

# Load Qwen2.5-Math 1.5B as the Computation Model
model_math, tokenizer_math = FastLanguageModel.from_pretrained(
    model_name = qwen_math_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_math)

# --- 3. FIX QWEN TOKENIZER (Manual ChatML Template) ---
# Qwen2.5 Base models require the ChatML template to function as an "Instruct" model
tokenizer_math.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

# --- 4. DEFINE PIPELINE ---
def collaborative_solve(question):
    # STAGE 1: Generate Solution Guidance (SG) using your Llama 1B
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Analyze the following math problem and provide a step-by-step solution guidance. Do not perform any numerical calculations; focus only on the logic and sequence of steps.

### Input:
{}

### Response:
{}"""

    inputs_sg = tokenizer_sg([alpaca_prompt.format(question, "")], return_tensors = "pt").to("cuda")
    outputs_sg = model_sg.generate(**inputs_sg, max_new_tokens = 150)
    guidance = tokenizer_sg.batch_decode(outputs_sg, skip_special_tokens = True)[0].split("### Response:")[1].strip()

    # STAGE 2: Computation with Qwen2.5-Math
    messages = [
        {"role": "system", "content": "You are a mathematical expert. Solve the problem by strictly following the provided logical strategy."},
        {"role": "user", "content": f"Problem: {question}\n\nStrategy: {guidance}"}
    ]

    # Apply the ChatML template
    prompt_math = tokenizer_math.apply_chat_template(messages, tokenize = False, add_generation_prompt = True)
    inputs_math = tokenizer_math([prompt_math], return_tensors = "pt").to("cuda")

    outputs_math = model_math.generate(
        **inputs_math,
        max_new_tokens = 512,
        repetition_penalty = 1.1, # Qwen is more stable than Llama, lower penalty needed
        temperature = 0.1,
        do_sample = True
    )

    decoded = tokenizer_math.batch_decode(outputs_math, skip_special_tokens = True)[0]

    # Extract only the assistant's part (after 'assistant')
    final_answer = decoded.split("assistant")[-1].strip()
    return guidance, final_answer

# --- 5. EXTENDED TEST SUITE ---
test_examples = [
    "A farm has 200 animals. 30% are cows, 50 are pigs, and the rest are goats. How many goats are there?",
    "The cheese pizza is cut into 12 slices and the pepperoni pizza is cut into 8 slices. If Kate's 6 friends each eat 6 cheese pizza slices and 4 pepperoni pizza slices, how many pizza pies does she need to buy?",
    "Bailey starts with $100. She receives $5 weekly allowance for 8 weeks. After 8 weeks, how much money does she have in total?",
    "A classroom whiteboard is shared by 4 teachers. Each teacher has 2 lessons a day. If the board is cleaned 3 times per lesson, how many times is it cleaned in a day?"
]

for i, example in enumerate(test_examples):
    print(f"\n" + "="*50)
    print(f"TEST CASE {i+1}: {example}")
    sg, answer = collaborative_solve(example)
    print(f"\n[1B LLAMA-SG GUIDANCE]:\n{sg}")
    print(f"\n[1.5B QWEN-MATH ANSWER]:\n{answer}")

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...


[1B LLAMA-SG GUIDANCE]:
Step 1: Determine the total number of slices consumed by all individuals.
Step 2: Calculate the number of pizza slices each person received.
Step 3: Find the total number of slices remaining after everyone has eaten.
Step 4: Divide the remaining slices by the number of slices in each pizza pie.

[1.5B QWEN-MATH ANSWER]:
Let's solve this step-by-step:

1. Total Slices Consumed:
   - Each friend eats 6 cheese pizza slices and 4 pepperoni pizza slices.
   - There are 6 friends, so the total number of cheese pizza slices eaten is \(6 \times 6 = 36\) slices.
   - The total number of pepperoni pizza slices eaten is \(6 \times 4 = 24\) slices.

2. Remaining Slices:
   - Cheese pizza slices remaining: \(12 - 36 = -24\)
   - Pepperoni pizza slices remaining: \(8 - 24 = -16\)

Since we cannot have negative slices, let's re-evaluate our steps:

- Correct Calculation for Cheese Pizza Slices:
  - Each friend eats 6 slices, but there are only 12 slices available initially. S

Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[1B LLAMA-SG GUIDANCE]:
Step 1: Identify the initial amount of money.
Step 2: Determine the total amount received as weekly allowances.
Step 3: Calculate the sum of all allowances to find the total money earned.
Step 4: Add the total earnings to the initial amount to find the final balance.

[1.5B QWEN-MATH ANSWER]:
Let's solve the problem step-by-step using Python code to ensure accuracy.

1. Identify the initial amount of money Bailey has.
2. Calculate the total amount of money Bailey receives from her weekly allowance over 8 weeks.
3. Add the total allowance to the initial amount to get the final balance.

Here is the Python code to perform these calculations:
```python
# Initial amount of money Bailey has
initial_amount = 100

# Weekly allowance and number of weeks
weekly_allowance = 5
number_of_weeks = 8

# Total amount received from weekly allowances
total_allowance = weekly_allowance * number_of_weeks

# Final balance after receiving the allowance
final_balance = initial_amount

Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[1B LLAMA-SG GUIDANCE]:
Step 1: Determine the total number of lessons taught across all teachers in a single day.
Step 2: Calculate the total number of times the board is cleaned based on the frequency of cleaning per lesson.

[1.5B QWEN-MATH ANSWER]:
Let's break down the problem step-by-step and use Python to ensure our calculations are accurate.

1. **Determine the total number of lessons taught across all teachers in a single day:**
   - Each teacher has 2 lessons a day.
   - There are 4 teachers.
   - Therefore, the total number of lessons in a day is \(4 \times 2 = 8\).

2. **Calculate the total number of times the board is cleaned based on the frequency of cleaning per lesson:**
   - The board is cleaned 3 times per lesson.
   - Therefore, the total number of cleanings in a day is \(8 \times 3 = 24\).

Now, let's implement this in Python to verify our solution.
```python
# Define the given values
teachers = 4
lessons_per_teacher_per_day = 2
cleanings_per_lesson = 3

# Calculate 